# Módulo 1 — Sistemas de classificação e NER
**Caso:** a *LojaAurora*, um e-commerce brasileiro, recebe milhares de reviews por dia. Ela quer
(1) separar automaticamente as reviews negativas para o SAC e (2) descobrir quais marcas e lugares aparecem nelas.

**Objetivo de aprendizagem:** comparar o desempenho de um sistema de ML clássico com um sistema de *deep learning* com *transformers*.

| Tempo | Etapa |
|---|---|
| 5 min | Dados (B2W-Reviews01, reviews reais da Americanas/Submarino) |
| 5 min | Baseline: TF-IDF + Regressão Logística |
| 10 min | Fine-tuning do BERTimbau |
| 5 min | Comparação e análise de erros |
| 5 min | NER: reconhecimento de entidades |

> Células com `# TODO` são para vocês completarem. Use `Shift+Enter` para rodar cada célula.

In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
MODO_RAPIDO = True   # True = subamostra para caber nos tempos da aula
SMOKE = os.environ.get("AULA_SMOKE") == "1"   # só para teste automatizado em CPU

if IN_COLAB:
    !pip install -q -U transformers datasets accelerate seqeval

import torch
import transformers
transformers.logging.set_verbosity_error()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| MODO_RAPIDO:", MODO_RAPIDO)
if DEVICE == "cpu" and not SMOKE:
    print("⚠️  Sem GPU. No Colab: Ambiente de execução → Alterar tipo → GPU T4.")

## 1. Os dados
O B2W-Reviews01 tem ~130 mil reviews com nota de 1 a 5. Vamos transformar isso num problema binário:
notas 1–2 = **negativo**, 4–5 = **positivo**. A nota 3 é ambígua e fica de fora.

In [ ]:
import numpy as np
import pandas as pd

URL = ("https://huggingface.co/datasets/ruanchaves/b2w-reviews01/resolve/"
       "refs%2Fconvert%2Fparquet/default/train/0000.parquet")
bruto = pd.read_parquet(URL, columns=["review_text", "overall_rating"])
print(bruto.shape)
bruto.dropna().sample(5, random_state=1)

In [ ]:
df = bruto.dropna(subset=["review_text"]).copy()
# TODO: descarte as notas 3 e crie a coluna `label` = 1 se overall_rating >= 4, senão 0
...
print(df.label.value_counts(normalize=True).round(3))

A base real é desbalanceada (bem mais reviews positivas). Para a aula vamos usar uma **amostra balanceada**,
o que deixa as métricas mais fáceis de ler. *Pergunta para a turma: o que muda em produção, onde a base não é balanceada?*

In [ ]:
from sklearn.model_selection import train_test_split

N_POR_CLASSE = 100 if SMOKE else (1500 if MODO_RAPIDO else 10000)
amostra = df.groupby("label").sample(N_POR_CLASSE, random_state=42)
treino, teste = train_test_split(amostra, test_size=0.2, stratify=amostra.label, random_state=42)
print(f"treino: {len(treino)} | teste: {len(teste)}")

## 2. Baseline: TF-IDF + Regressão Logística
Cada review vira um vetor esparso com o peso de cada palavra (e par de palavras). O modelo aprende um peso por termo.
Rápido, barato e explicável, e por isso sempre é o primeiro baseline.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, recall_score
from sklearn.pipeline import make_pipeline

t0 = time.time()
# TODO: monte um pipeline TfidfVectorizer(ngram_range=(1, 2), min_df=2) + LogisticRegression e treine com treino.review_text / treino.label
...
tempo_treino_base = time.time() - t0

t0 = time.time()
pred_base = baseline.predict(teste.review_text)
tempo_inf_base = time.time() - t0
print(f"Treino: {tempo_treino_base:.1f}s")
print(classification_report(teste.label, pred_base, target_names=["negativo", "positivo"], digits=3))

**Lendo as métricas**
- **Precisão (negativo):** das reviews que o modelo mandou para o SAC, quantas eram mesmo negativas? (custo: SAC ocupado à toa)
- **Recall (negativo):** das reviews negativas, quantas o modelo pegou? (custo: cliente insatisfeito ignorado)
- **F1:** média harmônica das duas. Útil quando as duas importam.

O baseline é explicável. Veja os termos que mais empurram para cada lado:

In [ ]:
vec = baseline.named_steps["tfidfvectorizer"]
clf = baseline.named_steps["logisticregression"]
termos, coefs = np.array(vec.get_feature_names_out()), clf.coef_[0]
print("Mais NEGATIVOS:", ", ".join(termos[coefs.argsort()[:15]]))
print("Mais POSITIVOS:", ", ".join(termos[coefs.argsort()[-15:]]))

## 3. Transformer: fine-tuning do BERTimbau
O [BERTimbau](https://huggingface.co/neuralmind/bert-base-portuguese-cased) é um BERT pré-treinado em português (110M parâmetros).
Ele já "sabe português". Só vamos ensinar a tarefa, colocando uma camada de classificação em cima e ajustando por **1 época**.

Primeiro, veja como o texto vira tokens (subpalavras):

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODELO = "neuralmind/bert-base-portuguese-cased"
tok = AutoTokenizer.from_pretrained(MODELO)
print(tok.tokenize("O produto chegou quebrado, péssimo atendimento! Não recomendo."))

def tokenizar(lote):
    return tok(lote["review_text"], truncation=True, max_length=128)

ds_treino = Dataset.from_pandas(treino[["review_text", "label"]], preserve_index=False).map(tokenizar, batched=True)
ds_teste = Dataset.from_pandas(teste[["review_text", "label"]], preserve_index=False).map(tokenizar, batched=True)

In [ ]:
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments)

modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO, num_labels=2,
    id2label={0: "negativo", 1: "positivo"}, label2id={"negativo": 0, "positivo": 1},
)
args = TrainingArguments(
    output_dir="bertimbau-reviews",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=3e-5,
    warmup_steps=10,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    max_steps=3 if SMOKE else -1,
)
trainer = Trainer(model=modelo, args=args, train_dataset=ds_treino,
                  data_collator=DataCollatorWithPadding(tok))

t0 = time.time()
trainer.train()
tempo_treino_bert = time.time() - t0
print(f"Treino: {tempo_treino_bert:.0f}s")

In [ ]:
t0 = time.time()
saida = trainer.predict(ds_teste)
tempo_inf_bert = time.time() - t0
pred_bert = saida.predictions.argmax(-1)
print(classification_report(teste.label, pred_bert, target_names=["negativo", "positivo"], digits=3))

## 4. Comparação: o transformer vale o custo?
Desempenho é só uma coluna. Num MBA a pergunta é **custo × benefício**.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

preds = {"TF-IDF + LogReg": pred_base, "BERTimbau": pred_bert}
tempos = {"TF-IDF + LogReg": (tempo_treino_base, tempo_inf_base),
          "BERTimbau": (tempo_treino_bert, tempo_inf_bert)}
parametros = {"TF-IDF + LogReg": clf.coef_.size + 1, "BERTimbau": modelo.num_parameters()}

comparacao = pd.DataFrame({
    nome: {
        "Acurácia": accuracy_score(teste.label, p),
        "F1 macro": f1_score(teste.label, p, average="macro"),
        "Recall negativo": recall_score(teste.label, p, pos_label=0),
        "Treino (s)": tempos[nome][0],
        "Inferência / 1.000 reviews (s)": tempos[nome][1] / len(teste) * 1000,
        "Parâmetros": f"{parametros[nome]:,}",
    }
    for nome, p in preds.items()
}).T
display(comparacao)

fig, eixos = plt.subplots(1, 2, figsize=(9, 3.5))
for eixo, (nome, p) in zip(eixos, preds.items()):
    ConfusionMatrixDisplay.from_predictions(teste.label, p, display_labels=["neg", "pos"], ax=eixo, colorbar=False)
    eixo.set_title(nome)
plt.tight_layout()

### Onde o transformer acerta e o baseline erra?
Repare em **negação**, **ironia** e **contraste** ("lindo, mas...").

In [ ]:
pd.set_option("display.max_colwidth", 220)
analise = teste.assign(base=pred_base, bert=pred_bert)
so_bert_acerta = analise[(analise.bert == analise.label) & (analise.base != analise.label)]
print(f"BERT acerta e baseline erra: {len(so_bert_acerta)} | o contrário: "
      f"{((analise.base == analise.label) & (analise.bert != analise.label)).sum()}")
so_bert_acerta[["review_text", "label"]].head(8)

In [ ]:
from transformers import pipeline

classificador = pipeline("text-classification", model=modelo, tokenizer=tok, device=0 if DEVICE == "cuda" else -1)

frases = [
    "Não é ruim, mas esperava mais pelo preço.",
    "Nada a reclamar, chegou antes do prazo!",
    "Lindo, só que parou de funcionar em uma semana.",
    # TODO: escrevam 2 frases difíceis para os modelos
]
for f in frases:
    base = ["negativo", "positivo"][baseline.predict([f])[0]]
    bert = classificador(f)[0]
    print(f"{f:55} | baseline: {base:8} | BERT: {bert['label']} ({bert['score']:.2f})")

**Para discutir (2 min):**
1. Em que cenário de negócio vocês ficariam com o TF-IDF? (volume, latência, explicabilidade, custo de GPU)
2. Ganhamos alguns pontos de F1. Quanto vale cada ponto de recall negativo para o SAC da LojaAurora?

## 5. NER: reconhecimento de entidades nomeadas
Agora a tarefa é classificar **cada token**: pessoa (PER), organização (ORG), local (LOC), data (DATE).
Vamos usar um modelo multilíngue já treinado para NER, sem fine-tuning.

In [ ]:
ner = pipeline("ner", model="Davlan/bert-base-multilingual-cased-ner-hrl",
               aggregation_strategy="simple", device=0 if DEVICE == "cuda" else -1)

exemplo = ("Comprei um notebook da Samsung nas Americanas em São Paulo e o João, "
           "da assistência técnica, resolveu tudo em março.")
for e in ner(exemplo):
    print(f"{e['entity_group']:5} {e['score']:.2f}  {e['word']}")

In [ ]:
from collections import Counter

textos = df.sample(40 if SMOKE else 300, random_state=0).review_text.str.slice(0, 1000).tolist()
# TODO: rode o `ner` na lista `textos` (dica: batch_size=16) e conte as entidades do tipo ORG em `orgs` (um Counter)
...
orgs.most_common(15)

### Métricas para NER
Em NER a métrica é **por entidade**, não por token. Uma entidade só conta como acerto se
**o início, o fim e o tipo** baterem. Acertar "Samsung" mas errar o fim de "Samsung Galaxy" é erro.
A biblioteca `seqeval` implementa isso:

In [ ]:
from seqeval.metrics import classification_report as relatorio_entidades

#          Samsung  Galaxy   chegou  em   São     Paulo   ,  João
gold = [["B-ORG", "I-ORG", "O",    "O", "B-LOC", "I-LOC", "O", "B-PER"]]
pred = [["B-ORG", "O",     "O",    "O", "B-LOC", "I-LOC", "O", "B-PER"]]
print(relatorio_entidades(gold, pred, digits=3))

## Para levar
- O baseline clássico é **rápido, barato e explicável**. O transformer ganha em **linguagem difícil** (negação, contexto).
- Precisão × recall é uma **decisão de negócio**, não técnica.
- O mesmo encoder (BERT) serve para classificar o texto inteiro **ou** cada token (NER). No Módulo 2 veremos por quê.